##Setup


Please note that Hugging Face losses use batched matching


In [ ]:
# @title
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass
from typing import Iterable, Sequence
from concurrent.futures import ThreadPoolExecutor
import csv
import hashlib
import importlib.metadata as metadata
import json
import math
import os
import platform
import random
import types
import warnings

os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='IProgress not found.*')
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Sampler
from scipy.optimize import linear_sum_assignment
from pycocotools import mask as mu
from transformers import Mask2FormerConfig, Mask2FormerForUniversalSegmentation
from transformers.models.mask2former import modeling_mask2former as m2f
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
ROOT = Path.cwd().resolve()
TRAIN_FROM_SCRATCH = False
PRETRAINED = 'facebook/mask2former-swin-small-coco-instance'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(4)
cv2.setNumThreads(1)
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print('Python', platform.python_version())
for package in ('torch', 'transformers', 'numpy', 'scipy', 'opencv-python-headless', 'pycocotools', 'safetensors'):
    print(package, metadata.version(package))
print('Device', DEVICE.type, '| inference float32 | single view | 2048 x 2048')

Python 3.13.16
torch 2.8.0
transformers 4.57.6
numpy 2.2.6
scipy 1.16.3
opencv-python-headless 4.12.0.88
pycocotools 2.0.10
safetensors 0.8.0
Device cuda | inference float32 | single view | 2048 x 2048


## Preprocessing


Unchanged MAGFiLO images (Ahmadzadeh et al., 2024) and GONG/NSO observations. [Source](https://www.kaggle.com/competitions/filament-segmentation-2026/data). [CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/).


In [ ]:
# @title
NATIVE = 2048
SPLIT_SEED = 2026
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)[:, None, None]
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)[:, None, None]
EXPECTED = dict(train=974, val=180)
@dataclass(frozen=True)
class Split:
    train_stems: list[str]
    val_stems: list[str]
    train_image_ids: list[str]
    val_image_ids: list[str]

    def summary(self) -> str:
        return f'train: {len(self.train_stems)} observations / {len(self.train_image_ids)} records | val: {len(self.val_stems)} observations / {len(self.val_image_ids)} records'

def instrument_of(file_name: str) -> str:
    return file_name.rsplit('.', 1)[0][-2:]

def split_records(annotation_path: str) -> tuple[dict[str, list[str]], dict[str, str]]:
    with open(annotation_path, encoding='utf-8') as fh:
        coco = json.load(fh)
    stem_to_ids: dict[str, list[str]] = defaultdict(list)
    id_to_stem: dict[str, str] = {}
    for record in coco['images']:
        stem_to_ids[record['file_name']].append(record['id'])
        id_to_stem[record['id']] = record['file_name']
    return (dict(stem_to_ids), id_to_stem)

def make_split(annotation_path: str, val_fraction: float=0.15, seed: int=SPLIT_SEED) -> Split:
    stem_to_ids, _ = split_records(annotation_path)
    by_site: dict[str, list[str]] = defaultdict(list)
    for stem in sorted(stem_to_ids):
        by_site[instrument_of(stem)].append(stem)
    rng = np.random.default_rng(seed)
    train_stems: list[str] = []
    val_stems: list[str] = []
    for site in sorted(by_site):
        stems = by_site[site]
        rng.shuffle(stems)
        n_val = max(1, round(len(stems) * val_fraction))
        val_stems.extend(stems[:n_val])
        train_stems.extend(stems[n_val:])
    train_stems.sort()
    val_stems.sort()
    return Split(train_stems=train_stems, val_stems=val_stems, train_image_ids=[i for s in train_stems for i in stem_to_ids[s]], val_image_ids=[i for s in val_stems for i in stem_to_ids[s]])

def assert_disjoint(split: Split) -> None:
    overlap = set(split.train_stems) & set(split.val_stems)
    if overlap:
        raise ValueError(f'{len(overlap)} observations appear in both folds: {sorted(overlap)[:5]}')

def _to_bytes(rle: dict) -> dict:
    counts = rle['counts']
    return dict(size=rle['size'], counts=counts.encode('ascii') if isinstance(counts, str) else counts)

def load_records(annotations: Path, fold: str) -> list[dict]:
    coco = json.loads(Path(annotations).read_text(encoding='utf-8'))
    meta = {record['id']: record for record in coco['images']}
    by_image: dict = defaultdict(list)
    for annotation in coco['annotations']:
        by_image[annotation['image_id']].append(annotation)
    split = make_split(str(annotations))
    assert_disjoint(split)
    image_ids = split.train_image_ids if fold == 'train' else split.val_image_ids
    if len(image_ids) != EXPECTED[fold]:
        raise SystemExit(f'canonical {fold} fold has {len(image_ids)} records, expected {EXPECTED[fold]}; the split has changed')
    records = []
    for image_id in image_ids:
        rles = []
        for annotation in by_image.get(image_id, []):
            segmentation = annotation.get('segmentation')
            if not segmentation:
                continue
            if isinstance(segmentation, list):
                rings = [ring for ring in segmentation if len(ring) >= 6]
                if not rings:
                    continue
                rle = mu.merge(mu.frPyObjects(rings, NATIVE, NATIVE))
            elif isinstance(segmentation['counts'], list):
                rle = mu.frPyObjects(segmentation, NATIVE, NATIVE)
            else:
                rle = segmentation
            rles.append(_to_bytes(rle))
        if rles:
            records.append(dict(image_id=str(image_id), file_name=meta[image_id]['file_name'], rles=rles, boxes=[mu.toBbox(rle).tolist() for rle in rles]))
    return records

def read_gray(path: Path) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None or image.shape != (NATIVE, NATIVE):
        raise ValueError('An image is unreadable or has the wrong dimensions')
    return image

def gpu_pixels(pixels: torch.Tensor, photometric: list | None, device: torch.device) -> torch.Tensor:
    x = pixels.to(device, non_blocking=True).float()
    if photometric is not None and any((item is not None for item in photometric)):
        if any((item is None for item in photometric)):
            raise ValueError('a micro-batch mixes augmented and plain samples')
        gamma = torch.tensor([item[0] for item in photometric], dtype=torch.float32, device=device).view(-1, 1, 1)
        gain = torch.tensor([item[1] for item in photometric], dtype=torch.float32, device=device).view(-1, 1, 1)
        x = torch.clamp(255.0 * (x / 255.0) ** gamma * gain, 0, 255)
    x = (x / 255.0)[:, None].expand(-1, 3, -1, -1)
    return (x - torch.as_tensor(MEAN, device=device)) / torch.as_tensor(STD, device=device)

class ExtentBatchSampler(Sampler):

    def __init__(self, records: int, crop: int, batch: int, whole_share: float, micro: int | None=None, crops_per_record: int=2, seed: int=2026):
        micro = micro or batch
        if batch % micro:
            raise ValueError(f'micro-batch {micro} must divide the step batch {batch}')
        self.records, self.crop, self.batch, self.micro = (records, crop, batch, micro)
        self.whole_batch = max(1, round(batch * (crop / NATIVE) ** 2))
        self.whole_micro = max(1, micro * self.whole_batch // batch)
        self.whole_share, self.seed, self.epoch = (whole_share, seed, 0)
        self.steps = records * crops_per_record // batch
        self.whole_steps = round(self.steps * whole_share)

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def __len__(self) -> int:
        crop_parts = math.ceil(self.batch / self.micro)
        whole_parts = math.ceil(self.whole_batch / self.whole_micro)
        return (self.steps - self.whole_steps) * crop_parts + self.whole_steps * whole_parts

    def __iter__(self):
        rng = np.random.default_rng([self.seed, self.epoch, 7])
        kinds = np.array([True] * self.whole_steps + [False] * (self.steps - self.whole_steps))
        rng.shuffle(kinds)
        stream: list[int] = []
        slot = 0
        for whole in kinds:
            size, micro = (self.whole_batch, self.whole_micro) if whole else (self.batch, self.micro)
            while len(stream) < size:
                stream.extend(rng.permutation(self.records).tolist())
            chosen, stream = (stream[:size], stream[size:])
            extent = NATIVE if whole else self.crop
            parts = math.ceil(size / micro)
            for part in range(parts):
                batch = []
                for record in chosen[part * micro:(part + 1) * micro]:
                    batch.append((record, extent, self.epoch, slot, part == parts - 1, parts))
                    slot += 1
                yield batch

class FilamentSamples(Dataset):

    def __init__(self, records: list[dict], images: Path, min_piece: int=40, seed: int=2026, augment: bool=True, rotate90: bool=True, preload: bool=True, cache: int=64, threads: int=8):
        self.records, self.images = (records, Path(images))
        self.min_piece, self.seed, self.augment, self.rotate90 = (min_piece, seed, augment, rotate90)
        self.preload, self.cache_size = (preload, cache)
        self._photos: dict[str, np.ndarray] = {}
        names = sorted({record['file_name'] for record in records})
        with ThreadPoolExecutor(max_workers=threads) as pool:
            if preload:
                self._photos = dict(zip(names, pool.map(lambda name: read_gray(self.images / name), names)))
            self._pieces = list(pool.map(self._record_pieces, records))

    @staticmethod
    def _record_pieces(record: dict) -> list[tuple[int, int, np.ndarray]]:
        pieces = []
        for rle in record['rles']:
            mask = mu.decode(rle)
            rows = np.flatnonzero(mask.any(axis=1))
            columns = np.flatnonzero(mask.any(axis=0))
            if len(rows) == 0:
                pieces.append((0, 0, np.zeros((0, 0), np.uint8)))
                continue
            y0, y1, x0, x1 = (int(rows[0]), int(rows[-1]) + 1, int(columns[0]), int(columns[-1]) + 1)
            pieces.append((x0, y0, np.ascontiguousarray(mask[y0:y1, x0:x1])))
        return pieces

    def __len__(self) -> int:
        return len(self.records)

    def _gray(self, name: str) -> np.ndarray:
        image = self._photos.get(name)
        if image is None:
            image = read_gray(self.images / name)
            if not self.preload:
                if len(self._photos) >= self.cache_size:
                    self._photos.pop(next(iter(self._photos)))
                self._photos[name] = image
        return image

    @staticmethod
    def _window(box, crop: int, rng, jitter: float) -> tuple[int, int]:
        x, y, w, h = box
        cx = x + w / 2 + rng.uniform(-jitter, jitter) * crop
        cy = y + h / 2 + rng.uniform(-jitter, jitter) * crop
        x0 = int(np.clip(round(cx - crop / 2), 0, NATIVE - crop))
        y0 = int(np.clip(round(cy - crop / 2), 0, NATIVE - crop))
        return (x0, y0)

    def _masks(self, record_index: int, x0: int, y0: int, crop: int) -> list[np.ndarray]:
        threshold = self.min_piece if crop < NATIVE else 1
        kept = []
        for bx, by, piece in self._pieces[record_index]:
            left, top = (max(bx, x0), max(by, y0))
            right, bottom = (min(bx + piece.shape[1], x0 + crop), min(by + piece.shape[0], y0 + crop))
            if left >= right or top >= bottom:
                continue
            window = piece[top - by:bottom - by, left - bx:right - bx]
            if int(window.sum()) < threshold:
                continue
            canvas = np.zeros((crop, crop), np.uint8)
            canvas[top - y0:bottom - y0, left - x0:right - x0] = window
            kept.append(canvas)
        return kept

    def __getitem__(self, key):
        record_index, extent, epoch, slot, last, parts = key
        record = self.records[record_index]
        rng = np.random.default_rng([self.seed, epoch, slot, record_index])
        gray = self._gray(record['file_name'])
        if extent >= NATIVE:
            x0 = y0 = 0
            masks = self._masks(record_index, 0, 0, NATIVE)
        else:
            masks = []
            for jitter in (0.35, 0.2, 0.0):
                anchor = record['boxes'][int(rng.integers(len(record['boxes'])))]
                x0, y0 = self._window(anchor, extent, rng, jitter)
                masks = self._masks(record_index, x0, y0, extent)
                if masks:
                    break
        if not masks:
            raise RuntimeError(f'no filament in sample of {record['image_id']} at {extent}')
        patch = gray[y0:y0 + extent, x0:x0 + extent]
        stack = np.stack(masks)
        photometric = None
        if self.augment:
            if rng.random() < 0.5:
                patch, stack = (patch[:, ::-1], stack[:, :, ::-1])
            if rng.random() < 0.5:
                patch, stack = (patch[::-1, :], stack[:, ::-1, :])
            if self.rotate90:
                turns = int(rng.integers(4))
                if turns:
                    patch, stack = (np.rot90(patch, turns), np.rot90(stack, turns, axes=(1, 2)))
            gamma = math.exp(rng.uniform(math.log(0.8), math.log(1.25)))
            gain = rng.uniform(0.85, 1.15)
            photometric = (gamma, gain)
        return (torch.from_numpy(np.ascontiguousarray(patch)), torch.from_numpy(np.ascontiguousarray(stack)), torch.zeros(len(stack), dtype=torch.long), (bool(last), int(parts), int(extent)), photometric)

def collate(batch):
    pixels = torch.stack([item[0] for item in batch])
    return (pixels, [item[1] for item in batch], [item[2] for item in batch], batch[0][3], [item[4] for item in batch])

def data_location():
    value = os.environ.get('SOL_DATA')
    if not value:
        return None
    base = Path(value)
    annotations = sorted(base.rglob('MAGFiLO_1.0_Annotations_kaggle2026_train.json'))
    images = sorted(p for p in base.rglob('train_images') if p.is_dir())
    if len(annotations) != 1 or len(images) != 1:
        raise ValueError('SOL_DATA must contain one training annotation file and one train_images folder')
    return annotations[0], images[0]

## Training (optional)


In [ ]:
# @title
IMAGE_SIZE = (2048, 2048)
mask_util = mu
@torch.inference_mode()
def predict(model, gray: np.ndarray, device: torch.device, bf16: bool, floor: float=0.02, min_pixels: int=40, chunk: int=8) -> list[tuple[float, np.ndarray]]:
    pixels = gpu_pixels(torch.from_numpy(np.ascontiguousarray(gray))[None], None, device)
    with torch.autocast(device.type, dtype=torch.bfloat16, enabled=bf16 and device.type == 'cuda'):
        output = model(pixel_values=pixels)
    probabilities = output.class_queries_logits[0].float().softmax(-1)[:, 0]
    logits = output.masks_queries_logits[0]
    keep = torch.nonzero(probabilities >= floor).flatten()
    candidates = []
    for start in range(0, len(keep), chunk):
        queries = keep[start:start + chunk]
        upsampled = F.interpolate(logits[queries][None].float(), size=gray.shape[:2], mode='bilinear', align_corners=False)[0]
        binary = upsampled > 0
        area = binary.flatten(1).sum(1)
        inside = (upsampled.sigmoid() * binary).flatten(1).sum(1) / area.clamp(min=1)
        scores = probabilities[queries] * inside
        binary = binary.to(torch.uint8).cpu().numpy()
        for j in range(len(queries)):
            if int(area[j]) >= min_pixels:
                candidates.append((float(scores[j]), binary[j]))
    return candidates

def paint_panoptic(candidates: Sequence[tuple[float, np.ndarray]], min_area: int=150, image_size: tuple[int, int]=IMAGE_SIZE) -> list[tuple[float, np.ndarray, int]]:
    ordered = sorted(candidates, key=lambda c: -c[0])
    claimed = np.zeros(image_size, dtype=np.uint8)
    kept: list[tuple[float, np.ndarray, int]] = []
    for score, binary in ordered:
        remaining = (binary.astype(np.uint8) & 1 - claimed).astype(np.uint8)
        area = int(remaining.sum())
        if area < min_area:
            continue
        claimed |= remaining
        kept.append((score, remaining, area))
    return kept

def _to_rle_objects(masks: Sequence[np.ndarray]) -> list[dict]:
    if not masks:
        return []
    stacked = np.stack([m.astype(np.uint8) for m in masks], axis=-1)
    return mask_util.encode(np.asfortranarray(stacked))

def compute_pq(pred_masks: Sequence[np.ndarray], gt_masks: Sequence[np.ndarray], iou_threshold: float=0.5) -> dict:
    n_pred, n_gt = (len(pred_masks), len(gt_masks))
    if n_pred == 0 or n_gt == 0:
        return {'pq': 0.0, 'sq': 0.0, 'rq': 0.0, 'tp': 0, 'fp': n_pred, 'fn': n_gt, 'iou_sum': 0.0}
    ious = np.asarray(mask_util.iou(_to_rle_objects(list(pred_masks)), _to_rle_objects(list(gt_masks)), [0] * n_gt)).reshape(n_pred, n_gt)
    matched_pred: set[int] = set()
    matched_gt: set[int] = set()
    iou_sum = 0.0
    pairs = np.argwhere(ious > iou_threshold)
    for i, j in sorted(pairs, key=lambda p: -ious[p[0], p[1]]):
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(int(i))
        matched_gt.add(int(j))
        iou_sum += float(ious[i, j])
    tp = len(matched_pred)
    fp = n_pred - tp
    fn = n_gt - tp
    sq = iou_sum / tp if tp else 0.0
    rq = tp / (tp + 0.5 * fp + 0.5 * fn) if tp + fp + fn else 0.0
    return {'pq': sq * rq, 'sq': sq, 'rq': rq, 'tp': tp, 'fp': fp, 'fn': fn, 'iou_sum': iou_sum}

def aggregate_pq(per_image: Iterable[dict]) -> dict:
    tp = fp = fn = 0
    iou_sum = 0.0
    for r in per_image:
        tp += r['tp']
        fp += r['fp']
        fn += r['fn']
        iou_sum += r['iou_sum']
    sq = iou_sum / tp if tp else 0.0
    rq = tp / (tp + 0.5 * fp + 0.5 * fn) if tp + fp + fn else 0.0
    return {'pq': sq * rq, 'sq': sq, 'rq': rq, 'tp': tp, 'fp': fp, 'fn': fn}

def encode(mask):
    rle = mu.encode(np.asfortranarray(mask.astype(np.uint8)))
    return dict(size=rle['size'], counts=rle['counts'].decode('ascii'))

def pq_rle(predicted, truth):
    matched_p, matched_g, total = (set(), set(), 0.0)
    if predicted and truth:
        ious = np.asarray(mu.iou(predicted, truth, [0] * len(truth)))
        pairs = np.argwhere(ious > 0.5)
        for p, g in sorted(pairs, key=lambda pair: -ious[pair[0], pair[1]]):
            if p not in matched_p and g not in matched_g:
                matched_p.add(int(p))
                matched_g.add(int(g))
                total += float(ious[p, g])
    tp = len(matched_p)
    return dict(tp=tp, fp=len(predicted) - tp, fn=len(truth) - tp, iou_sum=total)

def records_for(annotations):
    coco = json.loads(Path(annotations).read_text())
    split = make_split(str(annotations))
    assert_disjoint(split)
    record = {r['id']: r for r in coco['images']}
    by_id = defaultdict(list)
    val_ids = set(split.val_image_ids)
    for annotation in coco['annotations']:
        if annotation['image_id'] not in val_ids:
            continue
        r = record[annotation['image_id']]
        segmentation = annotation['segmentation']
        if isinstance(segmentation, list):
            rle = mu.merge(mu.frPyObjects(segmentation, r['height'], r['width']))
        elif isinstance(segmentation['counts'], list):
            rle = mu.frPyObjects(segmentation, r['height'], r['width'])
        else:
            rle = segmentation
        by_id[annotation['image_id']].append(rle)
    by_photo = defaultdict(list)
    for image_id in split.val_image_ids:
        by_photo[record[image_id]['file_name']].append((image_id, by_id[image_id]))
    return (dict(by_photo), split)

def evaluate_thresholds(cache, records, thresholds, grow=-1, min_area=300):
    thresholds = sorted(set(thresholds))
    totals = {c: [] for c in thresholds}
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    for name, passes in records.items():
        candidates = []
        for entry in cache[name]:
            if entry['score'] < thresholds[0]:
                continue
            mask = mu.decode(entry['rle'])
            if grow < 0:
                mask = cv2.erode(mask, kernel, iterations=-grow)
            elif grow != 0:
                raise ValueError(grow)
            candidates.append((entry['score'], mask))
        painted = [(score, encode(mask)) for score, mask, _ in paint_panoptic(candidates, min_area=min_area)]
        for conf in thresholds:
            predicted = [rle for score, rle in painted if score >= conf]
            for _, truth in passes:
                totals[conf].append(pq_rle(predicted, truth))
    return [dict(aggregate_pq(totals[c]), conf=c, grow=grow, min_area=min_area) for c in thresholds]

def _cost_matrix(matcher, masks_queries_logits, class_queries_logits, mask_labels, class_labels, point_coordinates):
    pred_probs = class_queries_logits.softmax(-1)
    pred_mask = masks_queries_logits
    cost_class = -pred_probs[:, class_labels]
    target_mask = mask_labels.to(pred_mask)
    target_mask = target_mask[:, None]
    pred_mask = pred_mask[:, None]
    target_coordinates = point_coordinates.repeat(target_mask.shape[0], 1, 1)
    target_mask = m2f.sample_point(target_mask, target_coordinates, align_corners=False).squeeze(1)
    pred_coordinates = point_coordinates.repeat(pred_mask.shape[0], 1, 1)
    pred_mask = m2f.sample_point(pred_mask, pred_coordinates, align_corners=False).squeeze(1)
    cost_mask = m2f.pair_wise_sigmoid_cross_entropy_loss(pred_mask, target_mask)
    cost_dice = m2f.pair_wise_dice_loss(pred_mask, target_mask)
    cost_matrix = matcher.cost_mask * cost_mask + matcher.cost_class * cost_class + matcher.cost_dice * cost_dice
    cost_matrix = torch.minimum(cost_matrix, torch.tensor(10000000000.0))
    cost_matrix = torch.maximum(cost_matrix, torch.tensor(-10000000000.0))
    return torch.nan_to_num(cost_matrix, 0)

def _sample_points(criterion, logits, oversampled, extra):
    num_boxes = logits.shape[0]
    num_points_sampled = int(criterion.num_points * criterion.oversample_ratio)
    point_coordinates = oversampled
    point_logits = m2f.sample_point(logits, point_coordinates, align_corners=False)
    point_uncertainties = criterion.calculate_uncertainty(point_logits)
    num_uncertain_points = int(criterion.importance_sample_ratio * criterion.num_points)
    num_random_points = criterion.num_points - num_uncertain_points
    idx = torch.topk(point_uncertainties[:, 0, :], k=num_uncertain_points, dim=1)[1]
    shift = num_points_sampled * torch.arange(num_boxes, dtype=torch.long, device=logits.device)
    idx += shift[:, None]
    point_coordinates = point_coordinates.view(-1, 2)[idx.view(-1), :].view(num_boxes, num_uncertain_points, 2)
    if num_random_points > 0:
        point_coordinates = torch.cat([point_coordinates, extra], dim=1)
    return point_coordinates

def _loss_masks(criterion, masks_queries_logits, mask_labels, indices, num_masks, oversampled, extra):
    src_idx = criterion._get_predictions_permutation_indices(indices)
    tgt_idx = criterion._get_targets_permutation_indices(indices)
    pred_masks = masks_queries_logits[src_idx]
    target_masks, _ = criterion._pad_images_to_max_in_batch(mask_labels)
    target_masks = target_masks[tgt_idx]
    pred_masks = pred_masks[:, None]
    target_masks = target_masks[:, None]
    with torch.no_grad():
        point_coordinates = _sample_points(criterion, pred_masks, oversampled, extra)
        point_labels = m2f.sample_point(target_masks, point_coordinates, align_corners=False).squeeze(1)
    point_logits = m2f.sample_point(pred_masks, point_coordinates, align_corners=False).squeeze(1)
    losses = {'loss_mask': m2f.sigmoid_cross_entropy_loss(point_logits, point_labels, num_masks), 'loss_dice': m2f.dice_loss(point_logits, point_labels, num_masks)}
    del pred_masks
    del target_masks
    return losses

def batched_loss_forward(self, masks_queries_logits, class_queries_logits, mask_labels, class_labels, auxiliary_predictions=None):
    layers = [(masks_queries_logits, class_queries_logits)]
    if auxiliary_predictions is not None:
        layers += [(aux['masks_queries_logits'], aux['class_queries_logits']) for aux in auxiliary_predictions]
    device = masks_queries_logits.device
    batch_size, num_queries = masks_queries_logits.shape[:2]
    matched = sum((min(num_queries, len(labels)) for labels in class_labels))
    oversampled_points = int(self.num_points * self.oversample_ratio)
    random_points = self.num_points - int(self.importance_sample_ratio * self.num_points)
    draws = []
    for _ in layers:
        matcher_points = [torch.rand(1, self.matcher.num_points, 2, device=device) for _ in range(batch_size)]
        oversampled = torch.rand(matched, oversampled_points, 2, device=device)
        extra = torch.rand(matched, random_points, 2, device=device) if random_points > 0 else None
        draws.append((matcher_points, oversampled, extra))
    with torch.no_grad():
        costs = [_cost_matrix(self.matcher, masks[i], classes[i], mask_labels[i], class_labels[i], points[i]) for (masks, classes), (points, _, _) in zip(layers, draws) for i in range(batch_size)]
        flat = torch.cat([cost.reshape(-1).float() for cost in costs]).cpu()
    indices_per_layer, offset = ([], 0)
    for layer in range(len(layers)):
        indices = []
        for i in range(batch_size):
            shape = costs[layer * batch_size + i].shape
            size = shape[0] * shape[1]
            rows, columns = linear_sum_assignment(flat[offset:offset + size].view(shape))
            offset += size
            indices.append((torch.as_tensor(rows, dtype=torch.int64), torch.as_tensor(columns, dtype=torch.int64)))
        indices_per_layer.append(indices)
    losses: dict[str, torch.Tensor] = {}
    for layer, ((masks, classes), (_, oversampled, extra), indices) in enumerate(zip(layers, draws, indices_per_layer)):
        num_masks = self.get_num_masks(class_labels, device=class_labels[0].device)
        part = {**_loss_masks(self, masks, mask_labels, indices, num_masks, oversampled, extra), **self.loss_labels(classes, class_labels, indices)}
        losses.update(part if layer == 0 else {f'{key}_{layer - 1}': value for key, value in part.items()})
    return losses

def install_batched_loss(model) -> None:
    model.criterion.forward = types.MethodType(batched_loss_forward, model.criterion)

def parameter_groups(model, lr: float, weight_decay: float, backbone_multiplier: float):
    norm_types = (torch.nn.LayerNorm, torch.nn.GroupNorm, torch.nn.BatchNorm2d)
    groups: dict[tuple[bool, bool], list] = {(b, d): [] for b in (True, False) for d in (True, False)}
    seen = set()
    for module_name, module in model.named_modules():
        for name, parameter in module.named_parameters(recurse=False):
            if not parameter.requires_grad or id(parameter) in seen:
                continue
            seen.add(id(parameter))
            full = f'{module_name}.{name}'
            backbone = full.startswith('model.pixel_level_module.encoder.')
            decay = not (isinstance(module, (*norm_types, torch.nn.Embedding)) or 'relative_position_bias_table' in name or 'absolute_pos_embed' in name)
            groups[backbone, decay].append(parameter)
    counts = {f'{('backbone' if b else 'head')}_{('decay' if d else 'plain')}': sum((p.numel() for p in params)) for (b, d), params in groups.items()}
    total = sum((p.numel() for p in model.parameters() if p.requires_grad))
    if sum(counts.values()) != total or not counts['backbone_decay'] or (not counts['head_decay']):
        raise SystemExit(f'parameter grouping is wrong: {counts}, total {total}')
    return ([dict(params=groups[b, d], lr=lr * (backbone_multiplier if b else 1.0), weight_decay=weight_decay if d else 0.0) for b, d in groups if groups[b, d]], counts)

def schedule(step: int, warmup: int, total: int) -> float:
    if step < warmup:
        return (step + 1) / warmup
    progress = min(1.0, (step - warmup) / max(1, total - warmup))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

def train_model(annotations, images, epochs=50, batch=8, max_steps=None, record_limit=None):
    torch.manual_seed(SEED)
    records = load_records(annotations, 'train')
    if record_limit is not None:
        records = records[:record_limit]
    dataset = FilamentSamples(records, images, preload=True)
    sampler = ExtentBatchSampler(len(records), 1024, batch, 0.25,
                                 micro=batch, crops_per_record=2, seed=SEED)
    loader = DataLoader(dataset, batch_sampler=sampler, collate_fn=collate, num_workers=0)
    net = Mask2FormerForUniversalSegmentation.from_pretrained(
        PRETRAINED, num_labels=1, ignore_mismatched_sizes=True).to(DEVICE)
    install_batched_loss(net)
    groups, _ = parameter_groups(net, 1e-4, 0.05, 0.1)
    optimizer = torch.optim.AdamW(groups, lr=1e-4, betas=(0.9, 0.999), fused=DEVICE.type == 'cuda')
    rates = [group['lr'] for group in optimizer.param_groups]
    total_steps = sampler.steps * epochs
    step, best = 0, -1.0
    destination = ROOT / 'trained_model'
    for epoch in range(epochs):
        sampler.set_epoch(epoch)
        net.train()
        losses = []
        for pixels, masks, classes, _, photometric in loader:
            pixels = gpu_pixels(pixels, photometric, DEVICE)
            masks = [item.to(DEVICE).float() for item in masks]
            classes = [item.to(DEVICE) for item in classes]
            with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=True):
                loss = net(pixel_values=pixels, mask_labels=masks, class_labels=classes).loss
            if not torch.isfinite(loss):
                raise ValueError('Training loss is not finite')
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 0.01)
            for group, rate in zip(optimizer.param_groups, rates):
                group['lr'] = rate * schedule(step, 500, total_steps)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            step += 1
            losses.append(float(loss.detach()))
            if max_steps is not None and step >= max_steps:
                net.float().save_pretrained(destination, max_shard_size='90MB')
                print('Training smoke steps', step, '| loss', round(losses[-1], 6))
                return destination
        if (epoch + 1) % 5 == 0 or epoch + 1 == epochs:
            score = training_validation(net, annotations, images)
            print('Epoch', epoch + 1, '| loss', round(float(np.mean(losses)), 6), '| validation PQ', round(score, 10))
            if score > best:
                best = score
                net.float().save_pretrained(destination, max_shard_size='90MB')
    return destination



def training_validation(net, annotations, images):
    records, _ = records_for(annotations)
    net.eval()
    cache = {}
    for name in sorted(records):
        cache[name] = [dict(score=score, rle=encode(mask)) for score, mask in
                       predict(net, read_gray(images / name), DEVICE, True, floor=0.05)]
    rows = []
    confs = tuple(round(0.05 * i, 2) for i in range(1, 20))
    for grow in (0, -1):
        for area in (100, 300):
            rows.extend(evaluate_thresholds(cache, records, confs, grow=grow, min_area=area))
    return max(row['pq'] for row in rows)


MODEL_FOLDER = ROOT / 'model'
if not os.environ.get('SOL_DATA'):
    print('Training skipped: SOL_DATA is unset')
elif TRAIN_FROM_SCRATCH:
    MODEL_FOLDER = train_model(*data_location())
else:
    print('Training skipped: stored final model selected')


Training skipped: stored final model selected


## Load model


In [ ]:
# @title
torch.backends.cuda.matmul.allow_tf32 = False
model = Mask2FormerForUniversalSegmentation.from_pretrained(
    MODEL_FOLDER, local_files_only=True, dtype=torch.float32).to(DEVICE).eval()
assert model.config.num_queries == 100
assert all(p.dtype == torch.float32 for p in model.parameters())
print('Model loaded locally | 100 queries | float32')

Model loaded locally | 100 queries | float32


## Inference


In [ ]:
# @title
test_images = sorted((ROOT / 'data/test').glob('*.jpeg'))
assert len(test_images) == 180, 'Expected 180 test images'

def inference(paths):
    for image_path in paths:
        yield image_path, predict(model, read_gray(image_path), DEVICE, False, floor=0.02)

## Post-processing


In [ ]:
# @title
def final_masks(candidates):
    selected = [(score, mask) for score, mask in candidates if score >= 0.85]
    return [mask for _, mask, _ in paint_panoptic(selected, min_area=100)]

##Write predictions and check against submission.csv


In [ ]:
# @title
count = 0
with (ROOT / 'predictions.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.writer(handle)
    writer.writerow(['filament_id', 'segmentation_rle'])
    for image_path, candidates in inference(test_images):
        for index, mask in enumerate(final_masks(candidates), 1):
            writer.writerow([f'{image_path.stem}_{index}', encode(mask)['counts']])
            count += 1
actual = (ROOT / 'predictions.csv').read_bytes()
reference = (ROOT / 'submission.csv').read_bytes()
print('Predicted instances', count)
print('Prediction SHA256', hashlib.sha256(actual).hexdigest())
assert actual == reference, 'Predictions differ from submission.csv'
print('predictions.csv matches submission.csv exactly')

Predicted instances 1192
Prediction SHA256 cca9c06eb7fc70e6fbff11609554a490f6f0a0bd34a48e88f648f434fc986bff
predictions.csv matches submission.csv exactly


## Evaluation


In [ ]:
# @title
if not os.environ.get('SOL_DATA'):
    print('Evaluation skipped: SOL_DATA is unset')
else:
    annotations, images = data_location()
    records = load_records(annotations, 'val')
    cache, results = {}, []
    for record in records:
        name = record['file_name']
        if name not in cache:
            cache[name] = final_masks(predict(model, read_gray(images / name), DEVICE, False, floor=0.02))
        results.append(compute_pq(cache[name], [mu.decode(rle) for rle in record['rles']]))
    result = aggregate_pq(results)
    print('Validation photographs', len(cache), '| annotation records', len(records))
    print('PQ', format(result['pq'], '.12f'), '| TP', result['tp'], '| FP', result['fp'], '| FN', result['fn'])
    print('Difference from recorded PQ', format(result['pq'] - 0.470524998967835, '+.12f'))

Validation photographs 106 | annotation records 180
PQ 0.470524998968 | TP 881 | FP 361 | FN 444
Difference from recorded PQ +0.000000000000
